# Demo 3: A Day-1 Early-Warning Cohort from Three Hospitals

Phase 3 of Dr. Reyes's winter program: Demo 2 showed that a patient can go hours without a monitor reading, so the program wants a score that flags deterioration early from routine ward data.

**This phase's ask.** In March, Dr. Reyes and the Chief Medical Information Officer, Dr. Daniel Kim, decide whether to pilot an early-warning score at the system's ward hospitals in New York, Chicago, and Phoenix. The score would run 24 hours after each admission and predict transfer to the ICU later in the stay. Dr. Kim asks the analytics team for the training table first: one row per 2024 ward encounter at hour 24, heart-rate features built only from earlier readings, lab results only if they were reported by then, and a later block of encounters held back to test on. The hospitals chart vitals on their own wall clocks, and the lab system stores times in UTC. All encounters, patients, and values are synthetic.

Run the cells from top to bottom.

In [ ]:
# Installs the course's pandas and pyarrow (for Parquet) in Colab (uv sync already did locally); if Colab asks, restart and rerun from the top
%pip install -q --no-warn-conflicts pandas==3.0.5 pyarrow==25.0.0

## Core walkthrough

### 1. Load the three hospitals' extracts

This supplied cell downloads `generate_data.py` when the folder lacks it (`urlretrieve(url, path)` saves the file at a web address to `path`), runs its `build()` when a table in `data/` is missing (about 20 seconds, once), and reads three tables. The seed is fixed, so your values match the ones below.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_RAW = 'https://raw.githubusercontent.com/christopherseaman/datasci_217/main/09/demo'
TABLES = ['encounters', 'ward_vitals', 'labs']
if not Path('generate_data.py').exists():
    urlretrieve(f'{REPO_RAW}/generate_data.py', 'generate_data.py')
if not all(Path(f'data/{name}.parquet').exists() for name in TABLES):
    import generate_data
    generate_data.build()

encounters = pd.read_parquet('data/encounters.parquet')
vitals = pd.read_parquet('data/ward_vitals.parquet')
labs = pd.read_parquet('data/labs.parquet')
print(encounters.shape, vitals.shape, labs.shape)
display(encounters.head(3))
display(vitals.head(3))
display(labs.head(3))

Expect `(10000, 8) (270306, 6) (57934, 5)`: 10,000 ward encounters in 2024, their charted heart rate and SpO2, and their lab orders. Each encounter has the patient's `age` and admitting `diagnosis`. `admitted_at` and `left_at` are already aware UTC datetimes (`datetime64[us, UTC]`). `recorded_local` is text on each hospital's wall clock, in no particular row order. The lab times are text ending in `Z`, which means UTC.

### 2. Three hospitals chart on three wall clocks

To order readings and compare them with UTC lab times, every reading needs one clock. `tz_localize()` attaches the zone each hospital charted in, and `tz_convert('UTC')` moves the readings to UTC. This intentional error shows the first attempt at the New York rows, which include the night the clocks fell back, November 3, 2024.

In [ ]:
ZONES = {'New York': 'America/New_York', 'Chicago': 'America/Chicago', 'Phoenix': 'America/Phoenix'}
new_york = vitals[vitals['site'] == 'New York']
try:
    pd.to_datetime(new_york['recorded_local'], format='%Y-%m-%d %H:%M').dt.tz_localize('America/New_York')
except ValueError as error:
    print('ValueError:', error)

Expect `ValueError: Cannot infer dst time from 2024-11-03 01:52:00, try using the 'ambiguous' argument`. That clock time happened twice, and the text does not say which. The fix sets these readings aside as `NaT`, along with any time that never happened, then converts to UTC. Each site is one piece, and `pd.concat()` (Lecture 06) stacks them.

In [ ]:
pieces = []
for site, zone in ZONES.items():
    piece = vitals[vitals['site'] == site].copy()
    local = pd.to_datetime(piece['recorded_local'], format='%Y-%m-%d %H:%M')
    piece['recorded_at'] = (local.dt.tz_localize(zone, ambiguous='NaT', nonexistent='NaT')
                            .dt.tz_convert('UTC'))
    pieces.append(piece)
localized = pd.concat(pieces)

set_aside = localized[localized['recorded_at'].isna()]
display(set_aside.groupby('site').size())
display(set_aside.sort_values('recorded_local')[['encounter_id', 'site', 'recorded_local']])

vitals = (localized.dropna(subset=['recorded_at'])
          .sort_values(['encounter_id', 'recorded_at'])
          .reset_index(drop=True))
print(vitals.shape, vitals['recorded_at'].dtype)

Expect `16` readings set aside: `12` in Chicago and `4` in New York, none in Phoenix, because Arizona does not change its clocks. The table lists `4` readings typed by hand at `2024-03-10 02:30` to `02:59`, an hour that never happened on the spring-forward night, and `12` charted between `2024-11-03 01:01` and `01:55`, the hour that happened twice. `vitals` keeps `(270290, 7)` rows, sorted by encounter and then time, in `datetime64[us, UTC]`.

**What we tell Dr. Kim:** 16 of 270,306 readings are set aside because their wall-clock time names no single instant; the table notes the count.

One Chicago patient, encounter `E08290`, was charted through that night.

In [ ]:
night = localized[(localized['encounter_id'] == 'E08290')
                  & localized['recorded_local'].between('2024-11-02 23:00', '2024-11-03 03:59')]
display(night.sort_values('recorded_local')[['recorded_local', 'recorded_at', 'heart_rate']])

Expect six rows. `00:51` becomes `05:51:00+00:00` and `03:56` becomes `09:56:00+00:00`: on the wall clock they look about 3 hours apart, but 4 hours passed. The `01:24` and `01:55` rows show `NaT`.

Plot the wall clock against UTC. The dotted lines show daylight time (UTC minus 5 hours) and standard time (UTC minus 6 hours). Each set-aside reading could sit on either line, so it gets an x-mark on both.

In [ ]:
import matplotlib.dates as mdates

night = night.assign(wall_clock=pd.to_datetime(night['recorded_local'])).sort_values('wall_clock')
kept = night.dropna(subset=['recorded_at'])
unsure = night[night['recorded_at'].isna()]
hours_utc = pd.date_range('2024-11-03 05:00', '2024-11-03 10:00', freq='h', tz='UTC')
fig, ax = plt.subplots(figsize=(8, 4))
for offset, style in [(5, ':'), (6, '--')]:
    ax.plot(hours_utc, hours_utc.tz_localize(None) - pd.Timedelta(hours=offset), style, color='gray',
            label=f'Wall clock = UTC minus {offset} hours')
ax.plot(kept['recorded_at'], kept['wall_clock'], 'o', color='#0072B2', markersize=8, label='Localized reading')
for offset in (5, 6):
    ax.plot((unsure['wall_clock'] + pd.Timedelta(hours=offset)).dt.tz_localize('UTC'), unsure['wall_clock'],
            'x', color='#D55E00', markersize=10, label='Set aside: two possible instants' if offset == 5 else None)
ax.axhspan(pd.Timestamp('2024-11-03 01:00'), pd.Timestamp('2024-11-03 02:00'), color='#D55E00', alpha=0.1)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.yaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set(title='Encounter E08290: the night Chicago fell back', xlabel='Time (UTC)', ylabel='Wall clock (Chicago)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.show()

Expect the three readings before `01:00` on the dotted daylight line and the `03:56` reading on the dashed standard line. The shaded band, `01:00` to `02:00` on the wall clock, crosses both lines, so those clock times map to two instants an hour apart: `01:24` could be `06:24` or `07:24` UTC.

### 3. The first feature draft hands one patient's heart rate to the next

The score's heart-rate features are the previous reading, its change, and recent means. The rows stack 10,000 encounters, and the first draft used a plain `shift(1)`, which runs straight down the column and hands each encounter's first row the previous encounter's last heart rate. Count the damage.

In [ ]:
first_row = ~vitals['encounter_id'].duplicated()
leaked = vitals['heart_rate'].shift(1)[first_row].notna().sum()
print('First rows:', first_row.sum(), ' given another encounter\'s heart rate:', leaked)

Expect `First rows: 10000  given another encounter's heart rate: 9866`. Grouping by `encounter_id` keeps every feature inside one stay, and each feature reads only earlier rows: `shift(1)` and `diff()` the previous reading, `shift(1)` before `rolling(3)` the three before, and `rolling('8h', closed='left')` the 8 hours before, without the current time.

In [ ]:
by_encounter = vitals.groupby('encounter_id')['heart_rate']
vitals['previous_hr'] = by_encounter.shift(1)
vitals['hr_change'] = by_encounter.diff()
vitals['mean_prev_3'] = by_encounter.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

prev_8h = (vitals.set_index('recorded_at')
           .groupby('encounter_id')['heart_rate']
           .rolling('8h', closed='left')
           .mean()
           .rename('mean_prev_8h')
           .reset_index())
vitals = vitals.merge(prev_8h, on=['encounter_id', 'recorded_at'], validate='one_to_one')
print(vitals.shape)
display(vitals.loc[vitals['encounter_id'] == 'E00002',
                   ['recorded_at', 'heart_rate', 'previous_hr', 'hr_change', 'mean_prev_3', 'mean_prev_8h']].head(5))

Expect `(270290, 11)`: the merge matched one feature row to each reading. In `E00002`'s first five readings, `previous_hr` and `hr_change` start as `NaN`. At `02:00` on January 2, `mean_prev_3` is `61.333333` (the 52, 62, and 70 readings, back to 14:02 the day before) but `mean_prev_8h` is `70.0`: only the 19:12 reading falls in the 8 hours before. The grouped rolling step takes a few seconds on 270,290 rows.

Plot the first 12 readings. At each reading's time, the gray step and the colored markers come only from the black dots to their left.

In [ ]:
first_12 = vitals[vitals['encounter_id'] == 'E00002'].head(12).set_index('recorded_at')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(first_12.index, first_12['heart_rate'], 'o', color='black', markersize=8, label='Charted heart rate')
ax.step(first_12.index, first_12['previous_hr'], where='pre', color='gray', label='previous_hr')
ax.plot(first_12.index, first_12['mean_prev_3'], 's', color='#0072B2', label='mean_prev_3')
ax.plot(first_12.index, first_12['mean_prev_8h'], '^', color='#D55E00', label='mean_prev_8h')
ax.set(title='Encounter E00002: past-only features', xlabel='Time (UTC)', ylabel='Heart rate (bpm)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid(alpha=0.3)
plt.show()

Expect the gray step to carry each black dot forward to the next reading time. At `02:00` on January 2 the blue square sits at `61.3`, the mean of the three dots before it, and the orange triangle sits at `70`, the only dot in the 8 hours before. No marker uses the dot directly above it.

### 4. Which labs had the score seen at hour 24?

The score runs 24 hours after admission, the **prediction time**. Encounters that ended before then (discharged or already transferred) never get a score, so they leave the cohort.

In [ ]:
encounters['prediction_time'] = encounters['admitted_at'] + pd.Timedelta(hours=24)
cohort = encounters[encounters['left_at'] > encounters['prediction_time']].copy()
print('Cohort:', len(cohort), 'of', len(encounters))
display(cohort['outcome'].value_counts())

Expect `Cohort: 9549 of 10000`, with `390` ICU transfers. `127` of the 517 transfers happened in the first 24 hours, before the score would run.

A lab result can feed the score only if it was reported by the prediction time, whenever its sample was drawn. `pd.to_datetime(..., utc=True)` reads the lab system's UTC text; a blank `resulted_at` (cancelled or never resulted) becomes `NaT`.

In [ ]:
labs['collected_at'] = pd.to_datetime(labs['collected_at'], utc=True)
labs['resulted_at'] = pd.to_datetime(labs['resulted_at'], utc=True)
labs['turnaround_h'] = (labs['resulted_at'] - labs['collected_at']).dt.total_seconds() / 3600
display(labs.groupby('test')['turnaround_h'].median().round(2))

cohort_labs = labs.merge(cohort[['encounter_id', 'prediction_time']], on='encounter_id', validate='many_to_one')
cohort_labs['collected_by_then'] = cohort_labs['collected_at'] <= cohort_labs['prediction_time']
cohort_labs['available'] = cohort_labs['resulted_at'] <= cohort_labs['prediction_time']
drawn = cohort_labs[cohort_labs['collected_by_then']]
known = pd.crosstab(drawn['test'], drawn['available'].map({True: 'reported by then', False: 'not yet reported'}))
display(known)

fig, ax = plt.subplots(figsize=(8, 3))
known[['reported by then', 'not yet reported']].plot(kind='barh', stacked=True, ax=ax, color=['#0072B2', '#D55E00'])
ax.set(title='Labs drawn before the prediction time', xlabel='Lab orders', ylabel='')
plt.show()

Expect median turnarounds of `0.67` hours for lactate, `1.15` for troponin, `1.80` for creatinine, and `45.23` for a blood culture. Among labs drawn by the prediction time, `not yet reported` counts `2136` blood cultures, `1949` creatinines, `49` lactates, and `56` troponins. In the chart, almost the whole blood-culture bar is orange (a culture takes about two days), and about a sixth of the creatinine bar.

**What we tell Dr. Kim:** blood cultures cannot feed a day-1 score, since almost none are back by hour 24. Lactate, troponin, and most creatinines can. Filtering on `collected_at` would have let 4,190 unreported results into the training table, and the score would test better than it can perform in the pilot.

### 5. One row per encounter at hour 24

The feature row for each encounter is its last reading at or before the prediction time. Sorting by encounter and time already happened, so `groupby().tail(1)` (Lecture 08) keeps the latest row. The latest lactate comes from the available results only.

In [ ]:
at_or_before = vitals.merge(cohort[['encounter_id', 'prediction_time']], on='encounter_id', validate='many_to_one')
at_or_before = at_or_before[at_or_before['recorded_at'] <= at_or_before['prediction_time']]
last_vitals = at_or_before.groupby('encounter_id').tail(1)

lactate = cohort_labs[(cohort_labs['test'] == 'lactate') & cohort_labs['available']]
last_lactate = (lactate.sort_values('resulted_at').groupby('encounter_id').tail(1)
                [['encounter_id', 'value']].rename(columns={'value': 'last_lactate'}))

features = (cohort[['encounter_id', 'patient_id', 'site', 'prediction_time', 'outcome']]
            .merge(last_vitals[['encounter_id', 'heart_rate', 'previous_hr', 'hr_change', 'mean_prev_3', 'mean_prev_8h']],
                   on='encounter_id', how='left', validate='one_to_one')
            .merge(last_lactate, on='encounter_id', how='left', validate='one_to_one'))
features['icu_transfer'] = features['outcome'] == 'ICU transfer'
print(features.shape)
display(features.head(3).round({'mean_prev_3': 1, 'mean_prev_8h': 1}))

Expect `(9549, 12)`: one row per encounter. A missing `last_lactate` means no lactate had been reported by the prediction time, not a normal result.

A first look at signal: the ICU-transfer rate by lactate group. `np.select()` (Lecture 03) names each group; a missing lactate matches neither condition, so it gets the default.

In [ ]:
features['lactate_group'] = np.select([features['last_lactate'] > 2, features['last_lactate'] <= 2],
                                       ['over 2', '2 or less'], default='none reported')
display(features.groupby('lactate_group')['icu_transfer'].agg(['count', 'mean']).round(3))

Expect transfer rates of `0.052` (`2005` encounters) with a lactate of 2 mmol/L or less, `0.131` (`1074`) over 2, and `0.022` (`6470`) with none reported.

**What we tell Dr. Kim:** a lactate over 2 mmol/L by hour 24 goes with 2.5 times the transfer rate of a lower one. Whether a lactate was ordered at all carries signal too, because a clinician orders one when worried, so the table keeps `none reported` as its own group.

### 6. Hold back the last quarter, the way the pilot will meet patients

The pilot will score future patients, so build on earlier encounters and test on the latest ones: a **chronological holdout** from October 1, 2024. This intentional error compares the aware `prediction_time` column with a naive cutoff.

In [ ]:
try:
    features['prediction_time'] < pd.Timestamp('2024-10-01')
except TypeError as error:
    print('TypeError:', error)

Expect `TypeError: Invalid comparison between dtype=datetime64[us, UTC] and Timestamp`: an aware column cannot be compared with a naive timestamp. Give the cutoff the column's zone.

In [ ]:
cutoff = pd.Timestamp('2024-10-01', tz='UTC')
features['block'] = np.where(features['prediction_time'] < cutoff, 'earlier', 'later_holdout')
display(features.groupby('block')['icu_transfer'].agg(['count', 'sum', 'mean']).round(3))
in_both = features.groupby('patient_id')['block'].nunique()
print('Patients in both blocks:', (in_both == 2).sum(), 'of', len(in_both))

Expect `7109` earlier encounters (`292` transfers) and `2440` in the holdout (`98`), both near a `0.040` transfer rate. `133` of 8,556 patients were readmitted across the cutoff and appear in both blocks.

**What we tell Dr. Kim:** the table is ready, with 7,109 encounters to build on and 2,440 later ones to test on. The holdout tests later encounters; 133 of its patients also appear in the earlier block.

### 7. Show the decision meeting one transfer

Dr. Kim wants one slide of what the score is meant to catch. `E05701`, a New York patient admitted with a GI bleed, was transferred on July 29. Draw its readings, a 4-hour rolling mean, and a dashed line at 100 bpm, the usual threshold for tachycardia (a fast heart rate). The axis label names the zone.

In [ ]:
one = vitals[vitals['encounter_id'] == 'E05701'].set_index('recorded_at')['heart_rate']
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(one.index, one, 'o', color='gray', markersize=4, label='Charted heart rate')
ax.plot(one.index, one.rolling('4h').mean(), color='#0072B2', linewidth=2, label='4-hour rolling mean')
ax.axhline(100, color='#D55E00', linestyle='--', label='Tachycardia (100 bpm)')
ax.set(title='Encounter E05701: heart rate before ICU transfer', xlabel='Time (UTC)', ylabel='Heart rate (bpm)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
plt.show()
display(encounters.loc[encounters['encounter_id'] == 'E05701', ['site', 'diagnosis', 'admitted_at', 'left_at', 'outcome']])

Expect readings every 4 hours between about 75 and 100 bpm for four days, then hourly checks on July 29 as the rolling mean crosses the dashed line and climbs to about 115 bpm before the transfer at `23:54` UTC.

## Independent practice

These are follow-up requests from the hospitals and reviewers; continue on your own after class. These cells reuse `vitals`, `last_vitals`, and `ZONES` from the core; if the runtime closed, run the cells above again first.

### 1. New York's daily count does not match the system report

The New York hospital counts its readings per day on its own clock, and its numbers differ from a report binned in UTC. Convert to the reporting zone before `resample('D')`; binning in UTC starts each New York day at 19:00 or 20:00 the evening before. `vitals` is sorted by encounter, not time, so sort the index before slicing dates.

In [ ]:
ny = vitals[vitals['site'] == 'New York'].set_index('recorded_at')['heart_rate'].sort_index()
utc_days = ny.loc['2024-11-01':'2024-11-05'].resample('D').count()
local_days = ny.tz_convert('America/New_York').loc['2024-11-01':'2024-11-05'].resample('D').count()
display(pd.DataFrame({'utc_day': utc_days.to_numpy(), 'new_york_day': local_days.to_numpy()},
                     index=['Nov 1', 'Nov 2', 'Nov 3', 'Nov 4', 'Nov 5']))
display(pd.DataFrame({'today': local_days, 'yesterday': local_days.shift(1, freq='D')}))

for day in ['2024-03-10', '2024-11-03', '2024-11-04']:
    hours = pd.date_range(day, pd.Timestamp(day) + pd.Timedelta(days=1), freq='h',
                          tz='America/New_York', inclusive='left')
    print(day, len(hours), 'hours')

Expect the columns to disagree every day, because a New York day starts 4 or 5 hours after a UTC day: `406` and `412` on November 1, `372` and `347` on November 2. November 3 differs most, `337` readings on the New York day against `298` on the UTC day: the shifted start accounts for most of the gap, and that local day also held one extra hour. The second table lines up each New York day with the day before by date: `shift(1, freq='D')` moves the labels a day later, so November 1 has no `yesterday` and a November 6 row holds November 5's `341`. The `date_range` lines print `23`, `25`, and `24` hours: a local daily count on March 10 covers one hour less.

### 2. A reviewer proposes the next heart rate as a feature

Two tempting features read the row after the current one: the next heart rate (a lead, `shift(-1)`) and a centered 3-reading mean. At each encounter's feature row from step 5, find when that next reading was charted.

In [ ]:
vitals['next_time'] = vitals.groupby('encounter_id')['recorded_at'].shift(-1)
check = (last_vitals[['encounter_id', 'recorded_at', 'prediction_time']]
         .merge(vitals[['encounter_id', 'recorded_at', 'next_time']],
                on=['encounter_id', 'recorded_at'], validate='one_to_one'))
hours_late = (check['next_time'] - check['prediction_time']).dt.total_seconds() / 3600
print('Feature rows:', len(check), ' next reading after the prediction time:', (hours_late > 0).sum())
print('Median hours after the prediction time:', round(hours_late.median(), 1))

Expect `Feature rows: 9549  next reading after the prediction time: 9428` and a median of `1.9` hours. The other 121 encounters had no later reading. Every feature row is the last reading before the score runs, so a lead or a centered window there reads a value nobody had yet. Use `shift(-1)` only to build a target, never a feature.

### 3. Phoenix asks why its offset never changes

The same zone name gives different UTC offsets in winter and summer, except in Phoenix: Arizona does not use daylight saving time.

In [ ]:
for zone in ZONES.values():
    winter = pd.Timestamp('2024-01-15 12:00', tz='UTC').tz_convert(zone)
    summer = pd.Timestamp('2024-07-15 12:00', tz='UTC').tz_convert(zone)
    print(f'{zone:<20} {winter.strftime("%H:%M %z")}  {summer.strftime("%H:%M %z")}')

Expect New York at `-0500` in January and `-0400` in July, Chicago at `-0600` and `-0500`, and Phoenix at `-0700` both times.